In [1]:
%pip -q install gradio cmake ninja pyngrok

In [ ]:
!nvidia-smi

In [ ]:
import os

os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["CUDACXX"] = "/usr/local/cuda/bin/nvcc"
os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on"
os.environ["FORCE_CMAKE"] = "1"

# Install llama-cpp-python using the pre-compiled CUDA wheels and 2 cores for compilation
%pip install --only-binary=:all: --no-cache-dir \
  llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu130 \
  --compile --config-settings=--global-option="--parallel=2"

In [ ]:
from google.colab import drive
import os
import sqlite3

drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/LocalLLM"
MODEL_PATH = os.path.join(
    BASE_DIR,
    "models",
    "qwen3-27b",
    "Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-IQ2_M.gguf",
)
CHAT_DB = os.path.join(BASE_DIR, "chat_history.sqlite3")

if not os.path.isfile(MODEL_PATH):
    raise FileNotFoundError(f"Model file not found in Google Drive: {MODEL_PATH}")
if not os.path.isfile(CHAT_DB):
    raise FileNotFoundError(f"Chat database not found in Google Drive: {CHAT_DB}")

with sqlite3.connect(CHAT_DB) as con:
    con.execute("""
        CREATE TABLE IF NOT EXISTS messages (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            conversation_id TEXT NOT NULL,
            role TEXT NOT NULL,
            content TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)

print("Model file:", MODEL_PATH)
print("Chat database:", CHAT_DB)

In [ ]:

import sqlite3
import uuid
import gradio as gr

SYSTEM_PROMPT = (
    "You are a helpful AI assistant. "
    "Give clear, accurate and thoughtful answers."
)

# ---------------- Database helpers ----------------

def get_conversations():
    with sqlite3.connect(CHAT_DB) as con:
        rows = con.execute("""
            SELECT conversation_id, MAX(id) AS last_id
            FROM messages
            GROUP BY conversation_id
            ORDER BY last_id DESC
        """).fetchall()
    return [row[0] for row in rows]


def get_messages(cid):
    if not cid:
        return []

    with sqlite3.connect(CHAT_DB) as con:
        rows = con.execute("""
            SELECT role, content
            FROM messages
            WHERE conversation_id = ?
            ORDER BY id
        """, (cid,)).fetchall()

    return [
        {"role": role, "content": content}
        for role, content in rows
        if role in ("user", "assistant")
    ]


def save_message(cid, role, content):
    with sqlite3.connect(CHAT_DB) as con:
        con.execute("""
            INSERT INTO messages (conversation_id, role, content)
            VALUES (?, ?, ?)
        """, (cid, role, content))


# ---------------- Chat actions ----------------

def new_conversation():
    cid = str(uuid.uuid4())[:8]
    choices = get_conversations()

    if cid not in choices:
        choices.insert(0, cid)

    return cid, [], gr.update(choices=choices, value=cid)


def load_conversation(cid):
    if not cid:
        return "", []
    return cid, get_messages(cid)


def chat(message, cid):
    message = (message or "").strip()
    cid = (cid or "").strip()

    if not message:
        return "", get_messages(cid), cid, gr.update(
            choices=get_conversations(), value=cid or None
        )

    if not cid:
        cid = str(uuid.uuid4())[:8]

    save_message(cid, "user", message)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ] + get_messages(cid)

    try:
        result = llm.create_chat_completion(
            messages=messages,
            temperature=0.7
        )

        answer = (
            result["choices"][0]["message"].get("content") or ""
        ).strip()

        if not answer:
            answer = "(The model returned an empty response.)"

        save_message(cid, "assistant", answer)

    except Exception as e:
        print("Model inference error:", repr(e))
        answer = f"Model error: {type(e).__name__}: {e}"

    choices = get_conversations()
    if cid not in choices:
        choices.insert(0, cid)

    return (
        "",
        get_messages(cid),
        cid,
        gr.update(choices=choices, value=cid)
    )


# ---------------- Gradio interface ----------------

with gr.Blocks(title="My AI Assistant") as demo:
    gr.Markdown("# My AI Assistant")
    gr.Markdown("Qwen3.8-27B · Chat history saved in SQLite")

    with gr.Row():
        cid = gr.Textbox(
            label="Conversation ID",
            placeholder="Create a new chat or select a saved one",
            scale=3
        )
        new_btn = gr.Button("New chat")

    saved_chats = gr.Dropdown(
        label="Saved conversations",
        choices=get_conversations(),
        interactive=True
    )

    chatbot_ui = gr.Chatbot(
    height=500,
    label="Chat"
    )

    message = gr.Textbox(
        label="Your message",
        placeholder="Message your AI...",
        lines=2
    )

    send_btn = gr.Button("Send", variant="primary")

    new_btn.click(
        new_conversation,
        outputs=[cid, chatbot_ui, saved_chats]
    )

    saved_chats.change(
        load_conversation,
        inputs=saved_chats,
        outputs=[cid, chatbot_ui]
    )

    send_btn.click(
        chat,
        inputs=[message, cid],
        outputs=[message, chatbot_ui, cid, saved_chats]
    )

    message.submit(
        chat,
        inputs=[message, cid],
        outputs=[message, chatbot_ui, cid, saved_chats]
    )

    demo.load(
        lambda: ("", []),
        outputs=[cid, chatbot_ui]
    )

print("Chat functions restored:", callable(chat))
print("Gradio interface restored:", demo is not None)


In [ ]:
import llama_cpp
from llama_cpp import Llama

if not llama_cpp.llama_supports_gpu_offload():
    raise RuntimeError(
        "llama-cpp-python was installed without CUDA support. "
        "Re-run the CUDA installation cell in a GPU-enabled Colab runtime."
    )

llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=-1,
    n_ctx=18432,
    verbose=False,
)
print("Qwen model loaded with GPU offload enabled.")

In [ ]:
import os, torch

# ---- 1. GPU memory (NVML = same numbers as nvidia-smi) ----
try:
    import pynvml
    pynvml.nvmlInit()
    info = pynvml.nvmlDeviceGetMemoryInfo(pynvml.nvmlDeviceGetHandleByIndex(0))
    vram_total, vram_used, vram_free = (x / 1024**2 for x in (info.total, info.used, info.free))
except Exception:
    free_b, total_b = torch.cuda.mem_get_info(0)
    vram_total, vram_free = total_b / 1024**2, free_b / 1024**2
    vram_used = vram_total - vram_free

# ---- 2. Model file size ----
model_size_mb = os.path.getsize(MODEL_PATH) / 1024**2 if os.path.exists(MODEL_PATH) else 0

# ---- 3. Current context ----
n_ctx = llm.n_ctx() if callable(llm.n_ctx) else llm.n_ctx

# ---- 4. Real KV cache cost from GGUF metadata ----
meta = llm.metadata
arch = meta["general.architecture"]
g = lambda k, d=None: meta.get(f"{arch}.{k}", d)

n_layers = int(g("block_count"))
n_head   = int(g("attention.head_count"))
n_kv     = g("attention.head_count_kv", n_head)
emb      = int(g("embedding_length"))
key_len  = int(g("attention.key_length", emb // n_head))
val_len  = int(g("attention.value_length", key_len))

# Number of KV heads summed over layers (handles per-layer lists and hybrid models)
if isinstance(n_kv, (list, tuple)) or hasattr(n_kv, "__len__") and not isinstance(n_kv, (str, int)):
    kv_heads_total = sum(int(x) for x in n_kv)
else:
    n_kv = int(n_kv)
    interval = g("full_attention_interval")          # hybrid models only
    attn_layers = n_layers // int(interval) if interval else n_layers
    kv_heads_total = n_kv * attn_layers

elems_per_token = kv_heads_total * (key_len + val_len)
bytes_per_token = {"f16": elems_per_token * 2.0, "q8_0": elems_per_token * 34/32}

kv_now_mb = {k: v * n_ctx / 1024**2 for k, v in bytes_per_token.items()}

# ---- 5. How much context could fit? ----
# If the model is already loaded, "free" is what's left AFTER the current KV cache.
reserve_mb = 600   # compute buffers, CUDA overhead, safety margin
extra_tokens = {k: max(0, (vram_free - reserve_mb) * 1024**2 / v) for k, v in bytes_per_token.items()}
max_ctx = {k: int(n_ctx + extra_tokens[k]) for k in bytes_per_token}

print("========== VRAM ANALYSIS ==========")
print(f"Total / Used / Free VRAM  : {vram_total:,.0f} / {vram_used:,.0f} / {vram_free:,.0f} MB")
print(f"Model file size           : {model_size_mb:,.0f} MB")
print("-----------------------------------")
print(f"Arch: {arch} | layers={n_layers} | kv_heads_total={kv_heads_total} | k/v dim={key_len}/{val_len}")
print(f"KV cost per token         : f16 {bytes_per_token['f16']/1024:,.1f} KB | q8_0 {bytes_per_token['q8_0']/1024:,.1f} KB")
print(f"KV cache at n_ctx={n_ctx:,}  : f16 {kv_now_mb['f16']:,.0f} MB | q8_0 {kv_now_mb['q8_0']:,.0f} MB")
print("-----------------------------------")
print(f"Max context that should fit (keeping {reserve_mb} MB spare):")
print(f"  with f16 KV cache       : ~{max_ctx['f16']:,} tokens")
print(f"  with q8_0 KV cache      : ~{max_ctx['q8_0']:,} tokens")
print("===================================")

In [ ]:
import os
import time
import socket
import json
import re
import threading
import traceback
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
import gradio as gr
from fastapi import FastAPI, Request, Header, HTTPException
from fastapi.responses import JSONResponse, StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from llama_cpp.llama_chat_format import Jinja2ChatFormatter
import uvicorn

# ---------------- 1. Read Kaggle Secrets (Add-ons -> Secrets) ----------------
# NGROK_AUTHTOKEN, NGROK_DOMAIN, and API_KEY are read from Kaggle Secrets.
# This keeps your credentials and API key completely confidential and out of the notebook source code.
_s = UserSecretsClient()
ngrok_token = _s.get_secret("NGROK_AUTHTOKEN")
ngrok_domain = _s.get_secret("NGROK_DOMAIN")

if not ngrok_token or not ngrok_domain:
    raise RuntimeError(
        "Add NGROK_AUTHTOKEN and NGROK_DOMAIN to Kaggle Secrets (Add-ons -> Secrets)."
    )

# Normalize domain
ngrok_domain = ngrok_domain.strip().rstrip("/")
ngrok_domain = (
    ngrok_domain
    .removeprefix("https://")
    .removeprefix("http://")
    .split("/")[0]
)

# Securely fetch API_KEY from Kaggle Secrets
try:
    API_KEY = _s.get_secret("API_KEY")
except Exception:
    API_KEY = None

if not API_KEY:
    API_KEY = os.environ.get("API_KEY", "sk-qwen36-kaggle-custom")
    print("=" * 60)
    print("⚠️  NOTE: 'API_KEY' was not found in Kaggle Secrets (Add-ons -> Secrets).")
    print(f"👉 Using temporary fallback key: {API_KEY}")
    print("👉 TO KEEP YOUR KEY SECRET: Go to Kaggle 'Add-ons' -> 'Secrets',")
    print("   create a secret named 'API_KEY' with your custom secret password/key.")
    print("=" * 60)
else:
    print("🔒 API_KEY loaded securely from Kaggle Secrets (kept private).")

# ---------------- 2. Configure Ngrok & Clean up previous runs ----------------
try:
    ngrok.set_auth_token(ngrok_token)
except Exception as e:
    if "Authtoken already set in configuration file" not in str(e):
        raise

# Terminate old tunnels and servers if re-running cell
try:
    ngrok.kill()
except Exception:
    pass

global uvicorn_server
if "uvicorn_server" in globals() and uvicorn_server is not None:
    uvicorn_server.should_exit = True
    time.sleep(1)

# ---------------- 3. Validate prerequisites ----------------
if "demo" not in globals():
    raise RuntimeError(
        "The variable 'demo' does not exist. Run the Gradio definition cell first."
    )

if "llm" not in globals():
    raise RuntimeError(
        "The variable 'llm' does not exist. Run your model-loading cell first."
    )

if "llm_lock" not in globals():
    llm_lock = threading.Lock()

# ---------------- 4. Define FastAPI with OpenAI Endpoints ----------------
api_app = FastAPI(
    title="Qwen3.6 Kaggle OpenAI Compatible Server",
    description="OpenAI-compatible endpoints for Continue (VS Code) & custom code editors",
    version="1.0.0"
)

api_app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

def verify_api_key(authorization: str = None, api_key_header: str = None, x_api_key: str = None, req_api_key: str = None):
    if not API_KEY:
        return True
    token = None
    if authorization:
        token = authorization.replace("Bearer ", "").strip()
    elif api_key_header:
        token = api_key_header.strip()
    elif x_api_key:
        token = x_api_key.strip()
    elif req_api_key:
        token = req_api_key.strip()
        
    if token != API_KEY:
        raise HTTPException(
            status_code=401,
            detail={
                "error": {
                    "message": "Invalid or missing API key. Provide header 'Authorization: Bearer YOUR_API_KEY'",
                    "type": "invalid_request_error",
                    "code": "invalid_api_key"
                }
            }
        )
    return True

def format_chat_prompt(messages):
    try:
        meta = llm.metadata
        eos = llm.detokenize([llm.token_eos()]).decode(errors="ignore")
        bos_id = llm.token_bos()
        bos = llm.detokenize([bos_id]).decode(errors="ignore") if bos_id >= 0 else ""
        tmpl = meta.get("tokenizer.chat_template")
        if tmpl:
            formatter = Jinja2ChatFormatter(
                template=tmpl,
                eos_token=eos, bos_token=bos, add_generation_prompt=True,
            )
            return formatter(messages=messages).prompt
    except Exception as e:
        print(f"Jinja chat template warning: {e}, falling back to ChatML standard")
    
    # Standard ChatML fallback format for Qwen models
    parts = []
    for m in messages:
        role = m.get("role", "user")
        content = m.get("content", "")
        parts.append(f"<|im_start|>{role}\n{content}<|im_end|>")
    parts.append("<|im_start|>assistant\n")
    return "\n".join(parts)

@api_app.get("/health")
def health():
    return {"status": "healthy", "model": "qwen3.6-35b-a3b", "vram": vram_usage()}

@api_app.get("/v1/models")
@api_app.get("/models")
async def list_models(request: Request, authorization: str = Header(None, alias="Authorization"), api_key_header: str = Header(None, alias="api-key"), x_api_key: str = Header(None, alias="x-api-key")):
    verify_api_key(authorization, api_key_header, x_api_key, request.query_params.get("api_key"))
    now = int(time.time())
    model_id = "qwen3.6-35b-a3b"
    return {
        "object": "list",
        "data": [
            {"id": model_id, "object": "model", "created": now, "owned_by": "kaggle-host", "permission": [], "root": model_id, "parent": None},
            {"id": "qwen3.6", "object": "model", "created": now, "owned_by": "kaggle-host", "permission": [], "root": model_id, "parent": None},
            {"id": "default", "object": "model", "created": now, "owned_by": "kaggle-host", "permission": [], "root": model_id, "parent": None}
        ]
    }

@api_app.post("/v1/chat/completions")
@api_app.post("/chat/completions")
async def chat_completions(request: Request, authorization: str = Header(None, alias="Authorization"), api_key_header: str = Header(None, alias="api-key"), x_api_key: str = Header(None, alias="x-api-key")):
    verify_api_key(authorization, api_key_header, x_api_key, request.query_params.get("api_key"))
    body = await request.json()

    raw_messages = body.get("messages", [])
    model_name = body.get("model", "qwen3.6-35b-a3b")
    temperature = float(body.get("temperature", 0.7))
    top_p = float(body.get("top_p", 0.95))
    stream = bool(body.get("stream", False))
    
    max_tokens = body.get("max_tokens") or body.get("max_completion_tokens")
    if max_tokens is not None:
        try:
            max_tokens = int(max_tokens)
        except (ValueError, TypeError):
            max_tokens = None

    stop = body.get("stop", ["<|im_end|>", "<|endoftext|>"])
    if isinstance(stop, str):
        stop = [stop, "<|im_end|>"]
    elif isinstance(stop, list):
        if "<|im_end|>" not in stop:
            stop.append("<|im_end|>")

    # Normalize messages for tokenization
    cleaned_messages = []
    for m in raw_messages:
        c = m.get("content", "")
        if isinstance(c, list):
            parts = []
            for p in c:
                if isinstance(p, dict) and p.get("type") == "text":
                    parts.append(p.get("text", ""))
                elif isinstance(p, str):
                    parts.append(p)
            c = "\n".join(parts)
        cleaned_messages.append({"role": m.get("role", "user"), "content": str(c or "")})

    cmpl_id = f"chatcmpl-{int(time.time()*1000)}"
    created_ts = int(time.time())

    # Tokenize using the proven Jinja2ChatFormatter / ChatML
    prompt_text = format_chat_prompt(cleaned_messages)
    prompt_tokens = llm.tokenize(prompt_text.encode("utf-8"), add_bos=False, special=True)

    # Context overflow protection: keep 2048 tokens reserved for completion
    max_ctx = llm.n_ctx() if callable(llm.n_ctx) else getattr(llm, "n_ctx", 32768)
    limit = max(1024, max_ctx - 2048)
    if len(prompt_tokens) > limit:
        print(f"Trimming prompt tokens from {len(prompt_tokens)} to {limit} to fit context.")
        prompt_tokens = prompt_tokens[-limit:]

    if not stream:
        with llm_lock:
            res_iter = llm.create_completion(
                prompt=prompt_tokens,
                max_tokens=max_tokens,
                temperature=temperature,
                top_p=top_p,
                stop=stop,
                stream=False
            )
        full_text = res_iter["choices"][0]["text"]
        full_text = re.sub(r"<think>.*?</think>", "", full_text, flags=re.S)
        full_text = re.sub(r"<think>.*$", "", full_text, flags=re.S).strip()
        return JSONResponse({
            "id": cmpl_id,
            "object": "chat.completion",
            "created": created_ts,
            "model": model_name,
            "choices": [{
                "index": 0,
                "message": {"role": "assistant", "content": full_text},
                "finish_reason": "stop"
            }],
            "usage": {
                "prompt_tokens": len(prompt_tokens),
                "completion_tokens": len(llm.tokenize(full_text.encode("utf-8"))),
                "total_tokens": len(prompt_tokens) + len(llm.tokenize(full_text.encode("utf-8")))
            }
        })

    # Streaming SSE generator with initial keepalive handshake
    def event_stream():
        try:
            # 1. Immediately send initial role chunk to open stream and prevent client timeout
            init_chunk = {
                "id": cmpl_id,
                "object": "chat.completion.chunk",
                "created": created_ts,
                "model": model_name,
                "choices": [{"index": 0, "delta": {"role": "assistant"}, "finish_reason": None}]
            }
            yield f"data: {json.dumps(init_chunk)}\n\n"

            with llm_lock:
                stream_iter = llm.create_completion(
                    prompt=prompt_tokens,
                    max_tokens=max_tokens,
                    temperature=temperature,
                    top_p=top_p,
                    stop=stop,
                    stream=True
                )
                
                in_think = False
                think_buf = ""
                
                for chunk in stream_iter:
                    text = chunk["choices"][0].get("text", "")
                    if not text:
                        continue
                    
                    if in_think:
                        think_buf += text
                        if "</think>" in think_buf:
                            remainder = think_buf.split("</think>", 1)[1].lstrip("\n")
                            in_think = False
                            think_buf = ""
                            if remainder:
                                delta_chunk = {
                                    "id": cmpl_id,
                                    "object": "chat.completion.chunk",
                                    "created": created_ts,
                                    "model": model_name,
                                    "choices": [{"index": 0, "delta": {"content": remainder}, "finish_reason": None}]
                                }
                                yield f"data: {json.dumps(delta_chunk)}\n\n"
                        continue
                    
                    if "<think>" in text:
                        pre, post = text.split("<think>", 1)
                        if pre:
                            delta_chunk = {
                                "id": cmpl_id,
                                "object": "chat.completion.chunk",
                                "created": created_ts,
                                "model": model_name,
                                "choices": [{"index": 0, "delta": {"content": pre}, "finish_reason": None}]
                            }
                            yield f"data: {json.dumps(delta_chunk)}\n\n"
                        if "</think>" in post:
                            remainder = post.split("</think>", 1)[1].lstrip("\n")
                            if remainder:
                                delta_chunk = {
                                    "id": cmpl_id,
                                    "object": "chat.completion.chunk",
                                    "created": created_ts,
                                    "model": model_name,
                                    "choices": [{"index": 0, "delta": {"content": remainder}, "finish_reason": None}]
                                }
                                yield f"data: {json.dumps(delta_chunk)}\n\n"
                        else:
                            in_think = True
                            think_buf = post
                        continue
                    
                    # Normal stream token
                    delta_chunk = {
                        "id": cmpl_id,
                        "object": "chat.completion.chunk",
                        "created": created_ts,
                        "model": model_name,
                        "choices": [{"index": 0, "delta": {"content": text}, "finish_reason": None}]
                    }
                    yield f"data: {json.dumps(delta_chunk)}\n\n"
            
            # Final stop chunk
            stop_chunk = {
                "id": cmpl_id,
                "object": "chat.completion.chunk",
                "created": created_ts,
                "model": model_name,
                "choices": [{"index": 0, "delta": {}, "finish_reason": "stop"}]
            }
            yield f"data: {json.dumps(stop_chunk)}\n\n"

        except Exception as e:
            print("Error in streaming generation:", repr(e))
            traceback.print_exc()
            err_chunk = {
                "id": cmpl_id,
                "object": "chat.completion.chunk",
                "created": created_ts,
                "model": model_name,
                "choices": [{"index": 0, "delta": {"content": f"\n\n[Model Error: {e}]"}, "finish_reason": "stop"}]
            }
            yield f"data: {json.dumps(err_chunk)}\n\n"
            
        yield "data: [DONE]\n\n"

    return StreamingResponse(
        event_stream(),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "Connection": "keep-alive", "X-Accel-Buffering": "no"}
    )

@api_app.post("/v1/completions")
@api_app.post("/completions")
async def text_completions(request: Request, authorization: str = Header(None, alias="Authorization"), api_key_header: str = Header(None, alias="api-key"), x_api_key: str = Header(None, alias="x-api-key")):
    verify_api_key(authorization, api_key_header, x_api_key, request.query_params.get("api_key"))
    body = await request.json()

    prompt = body.get("prompt", "")
    temperature = float(body.get("temperature", 0.7))
    top_p = float(body.get("top_p", 0.95))
    stream = bool(body.get("stream", False))
    
    max_tokens = body.get("max_tokens") or body.get("max_completion_tokens")
    if max_tokens is not None:
        try:
            max_tokens = int(max_tokens)
        except (ValueError, TypeError):
            max_tokens = None

    stop = body.get("stop", ["<|im_end|>", "<|endoftext|>"])
    if isinstance(stop, str):
        stop = [stop, "<|im_end|>"]
    elif isinstance(stop, list):
        if "<|im_end|>" not in stop:
            stop.append("<|im_end|>")

    if not stream:
        with llm_lock:
            res = llm.create_completion(
                prompt=prompt,
                temperature=temperature,
                top_p=top_p,
                max_tokens=max_tokens,
                stop=stop,
                stream=False
            )
        return JSONResponse(res)

    def stream_prompt():
        with llm_lock:
            for chunk in llm.create_completion(
                prompt=prompt,
                temperature=temperature,
                top_p=top_p,
                max_tokens=max_tokens,
                stop=stop,
                stream=True
            ):
                yield f"data: {json.dumps(chunk)}\n\n"
        yield "data: [DONE]\n\n"

    return StreamingResponse(
        stream_prompt(),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "Connection": "keep-alive"}
    )

# ---------------- 5. Mount Gradio & Start Background Server ----------------
# Gradio is mounted at root ("/") so the web interface remains accessible at https://<domain>/
api_app = gr.mount_gradio_app(api_app, demo, path="/")

class BackgroundServer(uvicorn.Server):
    def install_signal_handlers(self):
        pass

sock = socket.socket()
sock.bind(("127.0.0.1", 0))
port = sock.getsockname()[1]
sock.close()

print(f"Starting unified FastAPI + Gradio server on port {port}...")

config = uvicorn.Config(app=api_app, host="0.0.0.0", port=port, log_level="warning", loop="asyncio")
uvicorn_server = BackgroundServer(config=config)
server_thread = threading.Thread(target=uvicorn_server.run, daemon=True)
server_thread.start()

# Wait for server readiness
for _ in range(30):
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=2):
            print("Unified server is ready.")
            break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("Unified server failed to start.")

# ---------------- 6. Connect ngrok tunnel ----------------
try:
    tunnel = ngrok.connect(
        addr=port,
        proto="http",
        url="https://" + ngrok_domain
    )

    print("\n" + "=" * 65)
    print("🚀 SERVER ONLINE - GRADIO UI & OPENAI ENDPOINTS ACTIVE")
    print("=" * 65)
    print(f"🌐 Gradio Web UI        : {tunnel.public_url}")
    print(f"🤖 OpenAI API Base      : {tunnel.public_url}/v1")
    print(f"📦 Model Name in Config : qwen3.6-35b-a3b")
    print(f"🔑 API Key Status       : {'Active from Kaggle Secrets (kept confidential)' if _s.get_secret('API_KEY') else 'Fallback active: ' + API_KEY}")
    print("=" * 65)
    print(f"""
To configure Continue (in VS Code ~/.continue/config.yaml):

name: Main Config
version: 1.0.0
schema: v1
models:
  - name: Qwen3.6 Kaggle
    provider: openai
    model: qwen3.6-35b-a3b
    apiBase: {tunnel.public_url}/v1
    apiKey: {API_KEY}
    roles:
      - chat
      - edit
      - apply
    defaultCompletionOptions:
      contextLength: 32768
    requestOptions:
      timeout: 300000
      headers:
        ngrok-skip-browser-warning: "1"
""")

except Exception as e:
    print("Ngrok tunnel failed:", repr(e))
    raise

Ngrok authentication ready.
Starting Gradio on port: 34765
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

Gradio is ready.

CHATBOT ONLINE
Public URL: https://cherub-yo-yo-daffodil.ngrok-free.dev


In [ ]:
print("Model loaded:", "llm" in globals())
print("Chat functions loaded:", "chat" in globals())
print("Database path defined:", "CHAT_DB" in globals())
print("Gradio interface defined:", "demo" in globals())